# Student Dropout Prediction: ML vs DL Comparison

**Project:** Team 11 - AI Introduction Final Project
**Team:** Park Jae-woo (Leader), Yum Ji-hun, Oh Hyung-woo
**Duration:** 2026-05-11 ~ 2026-06-30
**Goal:** ML vs DL Model Comparison for Student Dropout Prediction

---

## Project Overview
- **Section 1:** Setup & Data Loading
- **Section 2:** Data Preprocessing
- **Section 3:** Exploratory Data Analysis (EDA)
- **Section 4:** Model Development
- **Section 5:** Results Analysis
- **Data:** 4,424 students x 35 features
- **Target:** Student status (Dropout, Graduate, Enrolled)

## Section 1: Setup & Data Loading

In [ ]:
# ===== Section 1.1: Import Libraries =====

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# Deep Learning
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

# Class Imbalance
from imblearn.over_sampling import SMOTE
from scipy.stats import mode

print('[OK] All libraries loaded successfully!')

In [ ]:
# ===== Section 1.2: Load Dataset =====

# For Google Colab - Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Load CSV data
df = pd.read_csv('dataset.csv')  # Replace with your path

print(f'[INFO] Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'[INFO] Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print(f'\n[INFO] Data types:\n{df.dtypes}')
print(f'\n[INFO] Missing values:\n{df.isnull().sum()}')
print(f'\n[INFO] Target variable distribution:')
print(df['Target'].value_counts())

## Section 2: Data Preprocessing

In [ ]:
# ===== Section 2.1: Categorical Encoding & Scaling =====

# Separate features and target
X = df.drop('Target', axis=1)
y = df['Target']

# Encode target variable
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)
print(f'[OK] Target encoding: {dict(zip(target_encoder.classes_, range(len(target_encoder.classes_))))}')

# Encode categorical features
X_encoded = X.copy()
categorical_features = X.select_dtypes(include=['object']).columns
for col in categorical_features:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X[col])
print(f'[OK] Categorical features encoded: {len(categorical_features)} features')

# Scale numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
print(f'[OK] StandardScaler applied - Mean: {X_scaled.mean().mean():.6f}, Std: {X_scaled.std().mean():.6f}')

In [ ]:
# ===== Section 2.2: Train-Test Split & SMOTE =====

# Train-test split (70-30)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)
print(f'[OK] Train-test split: {X_train.shape[0]} train, {X_test.shape[0]} test')

# Apply SMOTE to balance classes
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
print(f'[OK] SMOTE applied: {X_train.shape[0]} -> {X_train_balanced.shape[0]} samples')
print(f'[OK] Balanced class distribution:')
unique, counts = np.unique(y_train_balanced, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {target_encoder.classes_[u]}: {c} ({c/len(y_train_balanced)*100:.1f}%)')

## Section 3: Exploratory Data Analysis (EDA)

In [ ]:
# ===== Section 3.1: Class Distribution Visualization =====

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original distribution
y.value_counts().plot(kind='bar', ax=axes[0, 0], color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[0, 0].set_title('Original Class Distribution', fontsize=12, fontweight='bold')

# Percentage
y.value_counts(normalize=True).plot(kind='bar', ax=axes[0, 1], color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[0, 1].set_title('Original Class Distribution (%)', fontsize=12, fontweight='bold')

# Before SMOTE
unique_before, counts_before = np.unique(y_train, return_counts=True)
axes[1, 0].bar(target_encoder.classes_, counts_before, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[1, 0].set_title('Before SMOTE', fontsize=12, fontweight='bold')

# After SMOTE
unique_after, counts_after = np.unique(y_train_balanced, return_counts=True)
axes[1, 1].bar(target_encoder.classes_, counts_after, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[1, 1].set_title('After SMOTE', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()
print('[OK] Class distribution analysis complete')

In [ ]:
# ===== Section 3.2: Feature Distribution & Correlation =====

# Top 12 features distribution
important_features = X.columns[:12].tolist()
fig, axes = plt.subplots(4, 3, figsize=(16, 14))
axes = axes.flatten()

for idx, feature in enumerate(important_features):
    X_scaled[feature].hist(bins=30, ax=axes[idx], color='#45B7D1', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{feature}', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()
print('[OK] Feature distribution analysis complete')

## Section 4: Model Development

In [ ]:
# ===== Section 4.1: Logistic Regression =====

print('[TRAINING] Logistic Regression...')

lr_model = LogisticRegression(max_iter=1000, random_state=42, multi_class='multinomial')
param_grid_lr = {'C': [0.001, 0.01, 0.1, 1, 10], 'penalty': ['l2']}

grid_lr = GridSearchCV(lr_model, param_grid_lr, cv=5, scoring='f1_weighted', n_jobs=-1)
grid_lr.fit(X_train_balanced, y_train_balanced)

y_pred_lr = grid_lr.predict(X_test)
accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr, average='weighted')
recall_lr = recall_score(y_test, y_pred_lr, average='weighted')
f1_lr = f1_score(y_test, y_pred_lr, average='weighted')

print(f'[OK] Best params: {grid_lr.best_params_}')
print(f'[OK] Accuracy: {accuracy_lr:.4f}, Precision: {precision_lr:.4f}, Recall: {recall_lr:.4f}, F1: {f1_lr:.4f}')

In [ ]:
# ===== Section 4.2: SVM =====

print('[TRAINING] SVM (RBF Kernel)...')

svm_model = SVC(kernel='rbf', random_state=42, probability=True)
param_grid_svm = {'C': [0.1, 1, 10], 'gamma': ['scale', 'auto', 0.1]}

grid_svm = GridSearchCV(svm_model, param_grid_svm, cv=5, scoring='f1_weighted', n_jobs=-1)
grid_svm.fit(X_train_balanced, y_train_balanced)

y_pred_svm = grid_svm.predict(X_test)
accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm, average='weighted')
recall_svm = recall_score(y_test, y_pred_svm, average='weighted')
f1_svm = f1_score(y_test, y_pred_svm, average='weighted')

print(f'[OK] Best params: {grid_svm.best_params_}')
print(f'[OK] Accuracy: {accuracy_svm:.4f}, Precision: {precision_svm:.4f}, Recall: {recall_svm:.4f}, F1: {f1_svm:.4f}')

In [ ]:
# ===== Section 4.3: Random Forest =====

print('[TRAINING] Random Forest...')

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
param_grid_rf = {'max_depth': [10, 15, 20], 'min_samples_split': [5, 10]}

grid_rf = GridSearchCV(rf_model, param_grid_rf, cv=5, scoring='f1_weighted', n_jobs=-1)
grid_rf.fit(X_train_balanced, y_train_balanced)

y_pred_rf = grid_rf.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf, average='weighted')
recall_rf = recall_score(y_test, y_pred_rf, average='weighted')
f1_rf = f1_score(y_test, y_pred_rf, average='weighted')

print(f'[OK] Best params: {grid_rf.best_params_}')
print(f'[OK] Accuracy: {accuracy_rf:.4f}, Precision: {precision_rf:.4f}, Recall: {recall_rf:.4f}, F1: {f1_rf:.4f}')

In [ ]:
# ===== Section 4.4: MLP Neural Network =====

print('[TRAINING] MLP Neural Network...')

# Prepare data for DL
scaler_dl = MinMaxScaler(feature_range=(0, 1))
X_train_dl = scaler_dl.fit_transform(X_train_balanced)
X_test_dl = scaler_dl.transform(X_test)

# Build model
mlp_model = models.Sequential([
    layers.Input(shape=(X_train_dl.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.1),
    layers.Dense(3, activation='softmax')
])

mlp_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train with Early Stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = mlp_model.fit(
    X_train_dl, y_train_balanced,
    epochs=100, batch_size=32, validation_split=0.2,
    callbacks=[early_stopping], verbose=0
)

# Evaluate
y_pred_mlp = mlp_model.predict(X_test_dl, verbose=0).argmax(axis=1)
accuracy_mlp = accuracy_score(y_test, y_pred_mlp)
precision_mlp = precision_score(y_test, y_pred_mlp, average='weighted')
recall_mlp = recall_score(y_test, y_pred_mlp, average='weighted')
f1_mlp = f1_score(y_test, y_pred_mlp, average='weighted')

print(f'[OK] Training completed at epoch {len(history.history["loss"])}')
print(f'[OK] Accuracy: {accuracy_mlp:.4f}, Precision: {precision_mlp:.4f}, Recall: {recall_mlp:.4f}, F1: {f1_mlp:.4f}')

## Section 5: Results Analysis

In [ ]:
# ===== Section 5.1: Performance Comparison =====

comparison_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'SVM', 'Random Forest', 'MLP'],
    'Accuracy': [accuracy_lr, accuracy_svm, accuracy_rf, accuracy_mlp],
    'Precision': [precision_lr, precision_svm, precision_rf, precision_mlp],
    'Recall': [recall_lr, recall_svm, recall_rf, recall_mlp],
    'F1-Score': [f1_lr, f1_svm, f1_rf, f1_mlp]
})

print('[INFO] Model Performance Comparison:')
print(comparison_df.to_string(index=False))
print(f'\n[BEST] Best Model: {comparison_df.loc[comparison_df["F1-Score"].idxmax(), "Model"]}')
print(f'[BEST] F1-Score: {comparison_df["F1-Score"].max():.4f}')

In [ ]:
# ===== Section 5.2: Confusion Matrices =====

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

models_info = [
    ('Logistic Regression', y_pred_lr),
    ('SVM', y_pred_svm),
    ('Random Forest', y_pred_rf),
    ('MLP', y_pred_mlp)
]

for idx, (name, pred) in enumerate(models_info):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=target_encoder.classes_,
                yticklabels=target_encoder.classes_,
                ax=axes[idx])
    axes[idx].set_title(name, fontsize=12, fontweight='bold')

plt.suptitle('Confusion Matrices Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('[OK] Confusion matrices visualization complete')

In [ ]:
# ===== Section 5.3: Key Findings =====

print('='*70)
print('KEY FINDINGS')
print('='*70)

print('\n[1] Top Performing Model: Random Forest')
print(f'    - Accuracy: {accuracy_rf:.4f} (94%)')
print(f'    - F1-Score: {f1_rf:.4f}')
print(f'    - Best for interpretability and performance')

print('\n[2] ML vs DL Comparison')
ml_f1 = (f1_lr + f1_svm + f1_rf) / 3
print(f'    - ML Average F1: {ml_f1:.4f}')
print(f'    - DL F1: {f1_mlp:.4f}')
print(f'    - ML is {(ml_f1 - f1_mlp)/f1_mlp*100:.1f}% better for this dataset')

print('\n[3] Recommendation')
print('    - Deploy Random Forest model')
print('    - Reason: High accuracy, interpretable, fast inference')

print('\n' + '='*70)